In [ ]:
from pathlib import Path

import torch
from rtnls_enface.utils.plotting import plot_gridfns
from rtnls_inference import (
    HeatmapRegressionEnsemble,
    SegmentationEnsemble,
)

# noqa: F401
from rtnls_models.utils.config import load_config  # noqa: F401

## Segmentation of preprocessed images

Here we segment images preprocessed using 0_preprocess.ipynb


In [ ]:
ds_path = Path("../../samples/fundus")

# input folder. point to a folder with png and/or dicom images
rgb_path = ds_path / "preprocessed_rgb"
ce_path = ds_path / "preprocessed_ce"

# these are the output folders:
av_path = ds_path / "av"
vessels_path = ds_path / "vessels"
overlays_path = ds_path / "overlays"
discs_path = ds_path / "discs"
overlays_path = ds_path / "overlays"

device = torch.device("cuda:0")

### Artery-vein segmentation


In [ ]:
vessels_ensemble = SegmentationEnsemble.from_release("vessels_march_v2").to(device)
vessels_ensemble.predict(ds_path / "rgb", dest_path=vessels_path, num_workers=2)

In [ ]:
av_ensemble = SegmentationEnsemble.from_release("av_march_v2").to(device)
# run the model, pass the path for the outputs
av_ensemble.predict_preprocessed(rgb_path, ce_path, dest_path=av_path, num_workers=2)

### Disc segmentation


In [ ]:
disc_ensemble = SegmentationEnsemble.from_release("disc").to(device)
disc_ensemble.predict_preprocessed(
    rgb_path, ce_path, dest_path=discs_path, num_workers=2
)

### Fovea detection


In [ ]:
fovea_ensemble = HeatmapRegressionEnsemble.from_checkpoint("fovea_hm").to(device)
# note: this model does not use contrast enhanced images
df = fovea_ensemble.predict_preprocessed(rgb_path, num_workers=2)
df.columns = ["mean_x", "mean_y"]
df.to_csv(ds_path / "fovea.csv")

In [ ]:
df

### Plotting the retinas

This will only work if you ran all the models and stored the outputs using the same folder/file names as above


In [ ]:
from vascx.utils.loader import RetinaLoader

loader = RetinaLoader.from_folder(ds_path, fundus_subfolder="preprocessed_rgb")

In [ ]:
plot_gridfns([ret.plot for ret in loader[:8]])

## Storing visualizations


In [ ]:
if not overlays_path.exists():
    overlays_path.mkdir()
for ret in loader:
    fig, _ = ret.plot()
    fig.savefig(overlays_path / f"{ret.id}.png", bbox_inches="tight", pad_inches=0)